# HuggingFace 覆盖上传测试
验证：上传新 .pt 文件后，旧文件是否被真正删除（不留 LFS 历史）

In [ ]:
!pip install huggingface_hub torch -q

In [ ]:
import torch
from huggingface_hub import HfApi

REPO_ID = "wzmmmm/hf-overwrite-test"   # 测试用临时 repo，测完可删

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN 已从 Kaggle Secrets 读取")
except Exception:
    HF_TOKEN = None
    print("未找到 Kaggle Secrets，HF_TOKEN 为空")

api = HfApi(token=HF_TOKEN)

In [ ]:
# 创建测试 repo（已存在则跳过）
try:
    api.create_repo(repo_id=REPO_ID, repo_type="model", exist_ok=True, private=True)
    print(f"repo ready: {REPO_ID}")
except Exception as e:
    print(e)

In [ ]:
def upload_and_overwrite(api, repo_id, local_path, remote_name="model.pt"):
    """上传新文件，先删旧文件，再上传，再 squash 历史。"""
    # Step 1: 删除旧文件（不存在则跳过）
    try:
        api.delete_file(path_in_repo=remote_name, repo_id=repo_id)
        print(f"[1] 旧文件 {remote_name} 已删除")
    except Exception:
        print(f"[1] 旧文件不存在，跳过删除")

    # Step 2: 上传新文件
    api.upload_file(
        path_or_fileobj=local_path,
        path_in_repo=remote_name,
        repo_id=repo_id,
        commit_message=f"overwrite {remote_name}",
    )
    print(f"[2] 新文件上传完成")

    # Step 3: squash 历史，释放旧 LFS 对象
    api.super_squash_history(repo_id=repo_id)
    print(f"[3] 历史已压缩，旧 LFS 对象将被 GC")

In [ ]:
# 循环 10 次，每次生成不同参数的权重并覆盖上传
for step in range(1, 11):
    ckpt = {
        "step": step * 1000,
        "loss": round(1.0 - step * 0.05, 4),
        "weight": torch.randn(100, 100) * step,   # 每次不同
    }
    torch.save(ckpt, "/tmp/model.pt")
    print(f"\n=== step {step * 1000} ===")
    upload_and_overwrite(api, REPO_ID, "/tmp/model.pt")

In [ ]:
# 验证：查看 repo 当前文件列表和 commit 数
files = api.list_repo_files(repo_id=REPO_ID)
print("当前文件:", list(files))

commits = list(api.list_repo_commits(repo_id=REPO_ID))
print(f"当前 commit 数: {len(commits)}")
for c in commits:
    print(f"  {c.commit_id[:8]}  {c.title}")

In [ ]:
# 验证：下载回来确认是第二次的内容
from huggingface_hub import hf_hub_download
downloaded = hf_hub_download(repo_id=REPO_ID, filename="model.pt")
loaded = torch.load(downloaded)
print("下载回来的 step:", loaded["step"])   # 应为 2000
print("下载回来的 loss:", loaded["loss"])   # 应为 0.82

commits = list(api.list_repo_commits(repo_id=REPO_ID))
print(f"当前 commit 数: {len(commits)}")    # squash 后应为 1

In [ ]:
# ── 清理：删除测试 repo ──────────────────────────────────────
# 确认测试结果后取消注释
# api.delete_repo(repo_id=REPO_ID, repo_type="model")
# print("测试 repo 已删除")